# Binary (B1) storage precision: a 32× smaller vector payload, a rescore obligation that isn't optional

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/22-precision/binary-precision.ipynb)

Built from [`cookbook/book/chapters/22-precision/binary-precision.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/22-precision/binary-precision.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** the fourth `storage_precision` — `binary` (usearch's `B1` scalar kind, one
packed sign bit per dimension, searched by **Hamming** distance rather than cosine) —
set through `jammi.connect(..., config=...)` · the same retrieve→rescore path `f16`/`int8`
use, but resolved at `binary`'s own **precision-specific default oversample of 32** (vs. `4`
for every other precision) when the deployment leaves the knob unset · **Theory:** HNSW as
the navigable proximity graph every precision's sidecar is built over
(Malkov & Yashunin 2020), the recall-vs-memory trade quantization buys
(Johnson et al. 2021) · **Rail:** measurement (recall@1 *and* recall@10 against the engine's exact search, the
sidecar's on-disk byte count — decomposed into its vector payload and its HNSW link overhead —
and an independent numpy replay of the raw sign-bit Hamming ranking with no rescore at all;
every number measured live).

The [preceding chapter](https://f-inverse.github.io/jammi-ai/cookbook/chapters/22-precision/precision.html) measured `f16`/`int8`: both shrink the sidecar's own
stored vectors while leaving the retrieve→rescore default (`oversample = 4`) untouched, and
both recover the exact `f32` baseline's recall at that default. `binary` is the fourth point
on the same `storage_precision` axis, and it is not a smaller step in the same direction — it
is a much bigger one. Packing each dimension down to a single sign bit is the most aggressive
quantization the sidecar offers: **1 bit per dimension against `f32`'s 32**, a full 32× smaller
*vector payload*, searched by Hamming distance (a bit count) instead of cosine similarity. That
coarseness buys a large memory win on the structure a search actually traverses, but — measured
below — the sidecar *file*'s own total size shrinks by materially less than 32×, because an
HNSW graph also carries per-node link overhead that has nothing to do with how the vectors
themselves are stored, and that overhead does not shrink with `storage_precision` at all. The
aggressive quantization also means the retrieve stage's own ranking is far noisier than
`int8`'s — so `binary` needs its own, much wider default oversample to let the exact-`f32`
rescore recover the answer at all. This chapter measures both halves of that trade honestly: the
vector payload really does shrink 32×, the *whole graph file* shrinks by a real but smaller
factor once link overhead is counted, and the default oversample of `32` really is load-bearing,
not a conservative buffer — turn it down towards `1` and recall collapses, because the retrieve
stage no longer offers the rescore stage enough real candidates to recover from.

## The corpus, the oracle, and the tables

The same corpus the [preceding chapter](https://f-inverse.github.io/jammi-ai/cookbook/chapters/22-precision/precision.html) measures: the papers embedded once,
held-out queries, and the engine's exact search as the ground truth — both the true top-`k`
and its own top-1. Every table imports the same vectors.

In [ ]:
import numpy as np
from jammi_cookbook import contracts, precision, scale

SCALE = scale.current()
K = precision.K
corpus = precision.corpus(SCALE)
# Recall@1 moves a whole query at a time, and a coordinate at the sign boundary
# can quantize to either bit on another CPU: its tolerance admits one query's
# difference and never two.
AT_1_TOL = max(0.02, 1.5 / len(corpus.queries))
print(f"corpus {len(corpus.corpus_ids):,} papers × {corpus.dims} dims, "
      f"{len(corpus.queries)} held-out queries")

## The `f32` baseline — exact, single-stage

In [ ]:
with precision.built(corpus, "f32", None) as (db, table):
    f32_at_1 = precision.recall(db, corpus, k=1)
    f32_at_10 = precision.recall(db, corpus)
    f32_usearch = precision.bundle_bytes(db, table)["usearch"]
print(f"f32 recall@1 {f32_at_1:.4f}   recall@{K} {f32_at_10:.4f}   graph {f32_usearch:,} bytes")

## `binary` with the knob left unset — does the default land on 32?

A deployment that sets `storage_precision = "binary"` and never touches `oversample` must get
`binary`'s own default of `32`, not the `4` every other precision falls back to. On ONE table
built with no `oversample` key at all, the search with no override and the search with an
explicit per-request `32` must be the same search.

In [ ]:
with precision.built(corpus, "binary", None) as (db, table):
    default_at_1 = precision.recall(db, corpus, k=1)
    default_at_10 = precision.recall(db, corpus)
    explicit_at_1 = precision.recall(db, corpus, k=1, oversample=32)
    explicit_at_10 = precision.recall(db, corpus, oversample=32)
    binary_bytes = precision.bundle_bytes(db, table)
graph_ratio = binary_bytes["usearch"] / f32_usearch
print(f"binary, oversample unset:       recall@1 {default_at_1:.4f}   recall@{K} {default_at_10:.4f}")
print(f"same table, per-request 32:     recall@1 {explicit_at_1:.4f}   recall@{K} {explicit_at_10:.4f}")
print(f"binary graph {binary_bytes['usearch']:,} bytes ({graph_ratio:.1%} of f32's), "
      f"rescore companion {binary_bytes['rawf32']:,} bytes")

In [ ]:
assert (default_at_1, default_at_10) == (explicit_at_1, explicit_at_10), \
    "an unset oversample resolves to binary's own default of 32"
for metric, value in [("f32_at_1", f32_at_1), ("f32_at_10", f32_at_10),
                      ("binary_default_at_1", default_at_1), ("binary_default_at_10", default_at_10)]:
    contracts.assert_close(f"binary_precision.{metric}", value,
                           tol=AT_1_TOL if metric.endswith("_at_1") else 0.02)
contracts.assert_close("binary_precision.graph_ratio", graph_ratio, tol=0.03)

### The whole graph file is not 32× smaller

One bit per dimension against `f32`'s 32 shrinks the **vector payload** exactly 32×. But an
HNSW sidecar also stores each node's links — the navigable connectivity a search walks
(Malkov & Yashunin 2020) — and that overhead is a property of the graph's topology, not of how each
vector is stored. Subtracting the analytic vector payload from each measured file isolates it:

In [ ]:
rows = len(corpus.corpus_ids)
payload_f32 = rows * corpus.dims * 4
payload_binary = rows * ((corpus.dims + 7) // 8)
links_f32 = f32_usearch - payload_f32
links_binary = binary_bytes["usearch"] - payload_binary
print(f"{'':>8}{'vector payload':>16}{'link overhead':>16}{'file':>14}")
print(f"{'f32':>8}{payload_f32:>16,}{links_f32:>16,}{f32_usearch:>14,}")
print(f"{'binary':>8}{payload_binary:>16,}{links_binary:>16,}{binary_bytes['usearch']:>14,}")
print(f"payload ratio {payload_binary / payload_f32:.4f} — the 1-bit-vs-32-bit packing, exactly")
print(f"file ratio    {graph_ratio:.4f} — what a deployment keeps resident")

In [ ]:
assert abs(payload_binary / payload_f32 - 1 / 32) < 1e-9
assert graph_ratio > payload_binary / payload_f32, "link overhead does not shrink with precision"

"Binary is 32× smaller" is true of the vectors inside the graph; the file shrinks by the
measured, smaller factor above, because the links do not shrink at all. The smaller the
vectors, the larger the links' share: at a real embedding's hundreds of dimensions the payload
still dominates an `f32` graph, at the fixture encoder's 32 the links already weigh as much.
And, as for `int8`, a quantized table also writes the `.rawf32` rescore companion — the exact
vectors, which do not shrink with `storage_precision` at all.

## `binary` without rescore — an independent floor

`binary` always rescores: there is no knob, public or internal, that returns a raw Hamming
ranking as a search answer. To show why, numpy ranks the whole corpus by the Hamming distance
of a naive fixed-`0` sign code — bit = 1 iff the coordinate is positive — with no rescore.
(The engine's own code fits a per-dimension threshold instead; the
[next chapter](https://f-inverse.github.io/jammi-ai/cookbook/chapters/22-precision/asymmetric-binary.html) measures that fit.)

In [ ]:
corpus_bits = corpus.vectors > 0
hits_at_1, overlap_at_10 = 0, 0.0
for q, truth in zip(corpus.queries, corpus.exact):
    ranked = np.argsort(((q > 0) != corpus_bits).sum(axis=1), kind="stable")
    ids = [corpus.corpus_ids[j] for j in ranked[:K]]
    hits_at_1 += ids[0] == truth[0]
    overlap_at_10 += len(set(ids) & set(truth)) / K
hamming_at_1 = hits_at_1 / len(corpus.queries)
hamming_at_10 = overlap_at_10 / len(corpus.queries)
print(f"raw Hamming, no rescore:          recall@1 {hamming_at_1:.4f}   recall@{K} {hamming_at_10:.4f}")
print(f"engine, rescored at the default:  recall@1 {default_at_1:.4f}   recall@{K} {default_at_10:.4f}")

In [ ]:
contracts.assert_close("binary_precision.hamming_no_rescore_at_1", hamming_at_1, tol=0.0)
contracts.assert_close("binary_precision.hamming_no_rescore_at_10", hamming_at_10, tol=0.0)
assert hamming_at_1 < default_at_1 and hamming_at_10 < default_at_10, \
    "rescore recovers what the raw Hamming ranking loses"

A single sign bit per dimension, ranked by raw Hamming distance, is a coarse ranking; the
engine never returns it, and measuring it independently is what makes the sweep below legible:
`oversample` is not tuning a small correction, it is how much of that coarse ranking the exact
rescore gets to repair.

## Dialing `oversample` — binary needs a wider recovery than `int8`

The retrieve stage proposes `k * oversample` Hamming candidates for the exact rescore; a true
neighbour outside the candidate set cannot be recovered. Five otherwise-identical `binary`
tables, each stamped at an explicit `oversample`:

In [ ]:
GRID = [1, 4, 8, 16, 32]
sweep = {}
for ov in GRID:
    with precision.built(corpus, "binary", ov) as (db, _):
        sweep[ov] = (precision.recall(db, corpus, k=1), precision.recall(db, corpus))
print(f"{'oversample':>10}{'candidates':>12}{'recall@1':>11}{'recall@10':>11}")
for ov in GRID:
    print(f"{ov:>10}{K * ov:>12}{sweep[ov][0]:>11.4f}{sweep[ov][1]:>11.4f}")

In [ ]:
for ov in GRID:
    contracts.assert_close(f"binary_precision.os{ov}_at_1", sweep[ov][0], tol=AT_1_TOL)
    contracts.assert_close(f"binary_precision.os{ov}_at_10", sweep[ov][1], tol=0.02)
assert sweep[1][1] < sweep[32][1], "a narrow candidate set loses true neighbours"
assert all(sweep[a][1] <= sweep[b][1] + 1e-9 for a, b in zip(GRID, GRID[1:])), sweep

Unlike `int8`, whose recall reached the exact baseline by the shared default of `4`,
`binary`'s keeps climbing well past it: the coarser the retrieve ranking, the more candidates
the rescore needs to see. That is why `binary`'s own default is `32` — the point on this curve
where widening further stops buying recall.

## The per-request override — `search(..., oversample=...)`

On the table stamped at `oversample=1`, the same queries with a per-request `oversample=32`:

In [ ]:
with precision.built(corpus, "binary", 1) as (db, _):
    narrow = precision.recall(db, corpus)
    widened = precision.recall(db, corpus, oversample=32)
print(f"table default (1):            recall@{K} {narrow:.4f}")
print(f"per-request override to 32:   recall@{K} {widened:.4f}")

In [ ]:
assert widened == sweep[32][1], "a per-request 32 is the table-default 32's search"
assert widened > narrow

The override recovers, on the same on-disk graph and with no rebuild, the recall a wider table
default gives — reachable from `search`'s own keywords at `binary` precision too.

## Bridge note

> **The biggest quantization step needs the widest recovery, and rescore is what makes that
> honest.** HNSW (Malkov & Yashunin 2020) gives approximate search its speed by navigating a proximity
> graph instead of scanning every vector; `binary` pushes the accuracy-for-memory trade
> (Johnson et al. 2021) to its most aggressive point on this axis — one packed sign bit per
> dimension, ranked by Hamming distance, a **vector payload** a full 32× smaller than `f32`'s.
> Measured here: that 32× is real for the payload, but the *whole graph file* shrinks by a
> smaller, still-real factor, because a fixed per-node link-overhead cost (the topology HNSW
> itself needs, independent of how each node's vector is stored) does not shrink with
> `storage_precision` — so "binary is 32× smaller" is a claim about the vectors inside the
> graph, not the graph file a deployment actually keeps memory-resident. And the raw,
> un-rescored Hamming ranking really is close to useless on its own — but the engine never
> ships that number, because `needs_rescore()` is unconditional for `binary`. What the
> deployment default buys is not "32× cheaper, free" — it is "a much smaller vector payload
> inside the same graph topology, with recall recovered by a mandatory exact rescore at an
> oversample wide enough to still find the true neighbours in the coarse candidate set,"
> measured here at the engine's own default of `32` where an untouched deployment lands exactly
> there, not at the shared `4` every other precision uses. Both the deployment-default
> `storage_precision = "binary"` and the per-request `oversample` override are reachable from the
> same public `jammi.connect` / `search` surface every other chapter in this book writes against.

## References

- Malkov, Yu A., Yashunin, Dmitry A. (2020) *Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs* IEEE Transactions on Pattern Analysis and Machine Intelligence DOI 10.1109/TPAMI.2018.2889473; arXiv:1603.09320.
- Johnson, Jeff, Douze, Matthijs, Jégou, Hervé (2021) *Billion-Scale Similarity Search with GPUs* IEEE Transactions on Big Data DOI 10.1109/TBDATA.2019.2921572; arXiv:1702.08734.